# Main DLSTM: Final Model
## Data preparation and server check
### This notebook will not run any huge process. First you need to check dependencies based on <b>READEME</b> file. 

#### This model is redesigned based on research considerations and isn't an exact copy of the base article codes. In the "docs/Method_Decisions.md" you will find the key areas which are differ from base article.

In [ ]:
from pathlib import Path
import sys
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "thesis").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract the complete project folder; do not copy the notebook alone.")
if str(PROJECT) not in sys.path: 
    sys.path.insert(0, str(PROJECT))
from thesis.common import config
CONFIG = "configs/default.json"  # all notebooks must use the SAME config
c = config(CONFIG)
print("Experiment:", c["experiment_name"], "| outputs:", c["output_root"])


Experiment: MAIN_RECONSTRUCTION_v2 | outputs: C:\Users\behbodi.mohammad\Downloads\DLSTM_MAIN\DLSTM_MAIN\outputs\main


#### Checking environment attributes and network size:  

In [2]:
from thesis.pipeline import preflight
preflight(c)

{
  "hidden_widths": [
    4000,
    2000,
    2000,
    30
  ],
  "trainable_parameters": 144412092,
  "one_vector_GiB": 0.5379769653081894,
  "two_population_buffers_GiB": 215.19078612327576,
  "epochs": 30,
  "population": 200,
  "other_FE": 50000,
  "TAR_FE": 300000,
  "device": "cuda",
  "cuda_available": false,
  "notes": [
    "Disk-backed populations do not reduce candidate count or search dimensions.",
    "GPU activation/optimizer memory is additional; no runtime estimate without measuring the server.",
    "Hybrid uses 30-epoch Adam baseline then direct weight optimization; no training inside each fitness call.",
    "Population buffers are retained for resume and may consume substantial disk across folds/models."
  ]
}


{'hidden_widths': [4000, 2000, 2000, 30],
 'trainable_parameters': 144412092,
 'one_vector_GiB': 0.5379769653081894,
 'two_population_buffers_GiB': 215.19078612327576,
 'epochs': 30,
 'population': 200,
 'other_FE': 50000,
 'TAR_FE': 300000,
 'device': 'cuda',
 'cuda_available': False,
 'notes': ['Disk-backed populations do not reduce candidate count or search dimensions.',
  'GPU activation/optimizer memory is additional; no runtime estimate without measuring the server.',
  'Hybrid uses 30-epoch Adam baseline then direct weight optimization; no training inside each fitness call.',
  'Population buffers are retained for resume and may consume substantial disk across folds/models.']}

#### Creating completed sequences (Company-Years) which have atleast 100 days of trading data: 

In [ ]:
from thesis.data import prepare
import json, pandas as pd
prepared=prepare(c)
metadata = json.loads(
    (prepared / "metadata.json").read_text(encoding="utf-8")
)

print(json.dumps(metadata, ensure_ascii=False, indent=2))
samples=pd.read_csv(prepared/"samples.csv")
display(samples.groupby("year").agg(
    samples=("sample_id","size"),
    mean_days=("length","mean"),
    min_days=("length","min"),
    max_days=("length","max")
))

{
  "fingerprint": "7c92a40c8407b229",
  "input_sha256": "fb56a761cca7b2938cd7a06daf39ed3e3c145ca5c0a5b50b4ca0249d6f9e9fa7",
  "raw_rows": 409351,
  "samples": 1912,
  "retained_rows": 403439,
  "excluded_samples": 98,
  "features": [
    "بازده فروش",
    "بازده دارایی",
    "بازده سرمایه",
    "نسبت جاری",
    "نسبت آنی",
    "نسبت نقدینگی",
    "نسبت بدهی",
    "نسبت بدهی به سرمایه",
    "نسبت پوشش بدهی",
    "نسبت دارایی های جاری",
    "نسبت قیمت به درآمد",
    "نسبت قیمت به فروش",
    "بیشترین قیمت",
    "کمترین قیمت",
    "قیمت آخرین معامله",
    "قیمت باز شدن",
    "ارزش معاملات",
    "حجم معاملات"
  ],
  "min_days": 100,
  "max_days": 240,
  "label": "Already supplied: company-year t -> profitability t+1; no extra shift.",
  "information_availability": "Publication timestamps unavailable; forecast-time availability is an assumption, not verified."
}


,samples,mean_days,min_days,max_days
year,,,,
1389,105,176.523810,103,235
1390,108,185.361111,109,237
1391,106,182.707547,105,234
1392,133,198.872180,100,240
1393,131,206.267176,104,239
1394,127,203.480315,110,239
1395,132,206.053030,104,239
1396,132,214.083333,116,239
1397,134,222.970149,136,238


#### Recording necessary results on the notebook output:

In [4]:
from thesis.recovery import emit_capsule
emit_capsule([prepared/"metadata.json",prepared/"samples.csv",prepared/"excluded.csv"])

THESIS_CAPSULE_BEGIN ...
THESIS_CAPSULE_END
Embedded 3 essential files. Save notebook WITH outputs. Model weights/population buffers are not embedded.


#### Independent Backup cell
##### This cell will save the results based on notebook output cells. Wieghts and biases still need "model.pt" to resolve the previous results as backup. 
##### Also when <code>RESTORE = False</code> backup process will not be run and if you want to save a backup from outputs, you need to change it to <i><u>True</u></i>.

In [ ]:
RESTORE = False
NOTEBOOK_DIR = Path.cwd() if "Path" in globals() else __import__("pathlib").Path.cwd()
RECOVERY_DESTINATION = NOTEBOOK_DIR / "recovered_outputs"

def restore_from_notebooks(notebook_dir, destination):
    from pathlib import Path
    import base64,hashlib,json,re,zlib
    destination=Path(destination).resolve()
    destination.mkdir(parents=True,exist_ok=True)
    restored=[]
    for nb in sorted(Path(notebook_dir).rglob('*.ipynb')):
        notebook=json.loads(nb.read_text(encoding='utf-8'))
        texts=[]
        for cell in notebook.get('cells',[]):
            pieces=[]
            for output in cell.get('outputs',[]):
                text=output.get('text',output.get('data',{}).get('text/plain',''))
                pieces.append(''.join(text) if isinstance(text,list) else text)
            texts.append(''.join(pieces))
        for text in texts:
            for match in re.finditer(r'THESIS_CAPSULE_BEGIN ([a-f0-9]{64})\s+([A-Za-z0-9+/=\s]+?)THESIS_CAPSULE_END',text):
                raw=zlib.decompress(base64.b64decode(''.join(match[2].split())))
                if hashlib.sha256(raw).hexdigest()!=match[1]:
                    raise ValueError('Capsule checksum mismatch')
                
                archive=json.loads(raw)
                for name,item in archive['files'].items():
                    p=(destination/name).resolve()
                    if not p.is_relative_to(destination):
                        raise ValueError('Unsafe capsule path')
                    
                    b=base64.b64decode(item['data'])
                    if hashlib.sha256(b).hexdigest()!=item['sha256']:
                        raise ValueError('File checksum mismatch')
                    
                    if p.exists() and p.read_bytes()!=b:
                        # Preserve both historical snapshots; do not silently overwrite conflicting evidence.
                        p=p.with_name(p.name+'.snapshot_'+item['sha256'][:12])
                    p.parent.mkdir(parents=True,exist_ok=True)
                    p.write_bytes(b)
                    restored.append(str(p))
    print(f'Restored {len(set(restored))} files. Weights require model.pt/checkpoint backups.')
    return sorted(set(restored))

if RESTORE:
    restored = restore_from_notebooks(NOTEBOOK_DIR, RECOVERY_DESTINATION)
